In [22]:
import cv2 as cv
import numpy as np

In [23]:
yolo_path="/Users/yosefrezazadeh/Downloads/yolov3 (1).weights"
yolo_config="/Users/yosefrezazadeh/Downloads/yolov3 (1).cfg"

In [24]:
classes=[]
with open('/Users/yosefrezazadeh/Downloads/coco (1).names', 'r') as f:
  classes=[line.strip() for line in f.readlines()]

In [25]:
classes

['person',
 'bicycle',
 'car',
 'motorbike',
 'aeroplane',
 'bus',
 'train',
 'truck',
 'boat',
 'traffic light',
 'fire hydrant',
 'stop sign',
 'parking meter',
 'bench',
 'bird',
 'cat',
 'dog',
 'horse',
 'sheep',
 'cow',
 'elephant',
 'bear',
 'zebra',
 'giraffe',
 'backpack',
 'umbrella',
 'handbag',
 'tie',
 'suitcase',
 'frisbee',
 'skis',
 'snowboard',
 'sports ball',
 'kite',
 'baseball bat',
 'baseball glove',
 'skateboard',
 'surfboard',
 'tennis racket',
 'bottle',
 'wine glass',
 'cup',
 'fork',
 'knife',
 'spoon',
 'bowl',
 'banana',
 'apple',
 'sandwich',
 'orange',
 'broccoli',
 'carrot',
 'hot dog',
 'pizza',
 'donut',
 'cake',
 'chair',
 'sofa',
 'pottedplant',
 'bed',
 'diningtable',
 'toilet',
 'tvmonitor',
 'laptop',
 'mouse',
 'remote',
 'keyboard',
 'cell phone',
 'microwave',
 'oven',
 'toaster',
 'sink',
 'refrigerator',
 'book',
 'clock',
 'vase',
 'scissors',
 'teddy bear',
 'hair drier',
 'toothbrush']

In [26]:
yolo_net=cv.dnn.readNet(yolo_config, yolo_path)

In [27]:
layers=yolo_net.getLayerNames()
output_layers=[layers[i-1] for i in yolo_net.getUnconnectedOutLayers()]

In [28]:
colors = np.random.uniform(0, 255, size=(len(classes), 3))

In [29]:
cap = cv.VideoCapture("/Users/yosefrezazadeh/Downloads/traffic-mini (1).mp4")

frame_width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
video_cod = cv.VideoWriter_fourcc(*'MP4V')
video_output = cv.VideoWriter('driving_camera_det_yolov3_cv.mp4',
                      video_cod,
                      10,
                      (frame_width, frame_height))

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'


In [30]:
while(cap.isOpened()):
    ret, img = cap.read()
    # if not ret:  # اگر فریم خوانده نشد، حلقه را متوقف کن
    #     print("End of video or error reading frame.")
    #     break
    if ret == True:
        #img = cv.resize(frame, None, fx=0.8, fy=0.8)
        height, width, channels = img.shape
        #print(height, width)
        blob = cv.dnn.blobFromImage(img, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
        yolo_net.setInput(blob)
        outs = yolo_net.forward(output_layers)
        class_ids = []
        confidences = []
        boxes = []
        for out in outs:
            for detection in out:
                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                if confidence > 0.5:
                    # Object detected
                    center_x = int(detection[0] * width)
                    center_y = int(detection[1] * height)
                    w = int(detection[2] * width)
                    h = int(detection[3] * height)

                    # Rectangle coordinates
                    x = int(center_x - w / 2)
                    y = int(center_y - h / 2)

                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)

        indexes = cv.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)
        font = cv.FONT_HERSHEY_PLAIN
        for i in range(len(boxes)):
            if i in indexes:
                x, y, w, h = boxes[i]
                label = str(classes[class_ids[i]])
                color = colors[class_ids[i]]
                cv.rectangle(img, (x, y), (x + w, y + h), color, 2)
                cv.putText(img, label, (x, y + 30), font, 1, color, 1)
        cv.imshow('Frame',img)
        video_output.write(img)

        if cv.waitKey(25) & 0xFF == ord('q'):
            break
    else:
        break

cap.release()
video_output.release()
cv.destroyAllWindows()

2025-05-17 09:11:05.551 Python[11527:30539912] WARNING: Secure coding is not enabled for restorable state! Enable secure coding by implementing NSApplicationDelegate.applicationSupportsSecureRestorableState: and returning YES.
